In [54]:
import nfl_data_py as nfl
import pandas as pd
import os
class PbP:
    def __init__(self, team, year, category_path, output_folder="output"):
        self.team = team
        self.year = year
        self.output_folder = output_folder
        self.team_pbp = pd.DataFrame()
        self.category_path = category_path
        self.category_data = dict()

    def set_pbp_and_categories(self):
        pbp = nfl.import_pbp_data(years = [self.year])
        team_p = pbp[(pbp["posteam"] == self.team) | (pbp["defteam"] == self.team)]
        team_p = team_p.fillna(0)
        self.team_pbp = team_p
    def category_datasets(self):
        column_categories = pd.read_csv(self.category_path)
        categories = column_categories.groupby("category").agg(list).reset_index()
        for i in range(categories.shape[0]):
            if i == 0:
                continue
            category = categories.iloc[i][0]
            cols = categories.iloc[i][1]
            if (self.year > 2015):
                if "Identifiers" in category:
                    cols.remove("old_game_id")
            cur_df = self.team_pbp[cols]
            self.category_data[category] = cur_df
            folder_path = self.output_folder
            if not os.path.exists(folder_path):
                os.makedirs(folder_path)
            file_name = category + "1.csv"
            path = os.path.join(folder_path, file_name)
            cur_df.to_csv(path, index=False)
        #other columns in newer datasets
        if (self.year > 2015):
            pbp2 = nfl.import_pbp_data(years = [2015])
            new_columns = list(set(self.team_pbp.columns) - set(pbp2.columns))
            advanced_features = self.team_pbp[new_columns]
            file_name2= "Advanced_features1.csv"
            path = os.path.join(folder_path, file_name2)
            cur_df.to_csv(path, index=False)
    def weekly_stats_by_type(self,outcomes,type,probabilities):
        sum_stats = []
        weekly_stats = pd.DataFrame()
        average_stats = list(probabilities.columns)
        average_stats.append("ydstogo")
        not_summed = ["penalty_player_id", "penalty_player_name","replay_or_challenge", "replay_or_challenge_result", "penalty_type", "ydstogo", "ydsnet", "desc"]
        cols = list(outcomes.columns)
        for col in cols:
            if col in not_summed:
                continue
            else:
                sum_stats.append(col)
        if type == 'passing':
            sum_stats[0] = 'passing_yards'
            passers = self.team_pbp[self.team_pbp["passer_player_name"] != 0]
            weekly_stats = (
                passers.groupby(['passer_player_name',"play_type",'week', 'posteam'])[sum_stats].sum()
                .join(passers.groupby(['passer_player_name',"play_type",'week', 'posteam'])[average_stats].mean())
                .reset_index()
            )
        elif type == 'rushing':
            sum_stats[0] = 'rushing_yards'
            rushers = self.team_pbp[self.team_pbp["rusher_player_name"] != 0]
            weekly_stats = (
                rushers.groupby(['rusher_player_name',"play_type",'week', 'posteam'])[sum_stats].sum()
                .join(rushers.groupby(['rusher_player_name',"play_type",'week', 'posteam'])[average_stats].mean())
                .reset_index()
            )
        else:
            sum_stats[0] = 'receiving_yards'
            receivers = self.team_pbp[self.team_pbp["receiver_player_name"] != 0]
            weekly_stats = (
                receivers.groupby(['receiver_player_name',"play_type",'week', 'posteam'])[sum_stats].sum()
                .join(receivers.groupby(['receiver_player_name',"play_type",'week', 'posteam'])[average_stats].mean())
                .reset_index()
            )
        #Split into Ind and opp stats
        team_stats = weekly_stats[weekly_stats["posteam"] == self.team].sort_values(by = "week").reset_index(drop=True)
        opp_stats = weekly_stats[weekly_stats["posteam"] != self.team].sort_values(by = "week").reset_index(drop=True)
        #convert to csv
        team_stats.to_csv(self.output_folder + "/" + self.team + type + "stats1.csv")
        opp_stats.to_csv(self.output_folder + "/opp" + type + "stats1.csv")
    def evaluate(self):
        self.set_pbp_and_categories()
        self.category_datasets()
        probabilities = [self.category_data[key] for key in self.category_data if "Probabilities" in key][0]
        outcomes = [self.category_data[key] for key in self.category_data if "Outcomes" in key][0]
        types = ["passing", "rushing", "receiving"]
        for t in types:
            self.weekly_stats_by_type(outcomes, t, probabilities)
        





        


    

In [ ]:
"Personal project/nfl_pbp_column_categories_v3 (1).csv"

In [37]:
#test
colts_2004 = PbP("IND", 2004, "Personal project/nfl_pbp_column_categories_v3 (1).csv", "playbyplay")

In [38]:
colts_2004.evaluate()

2004 done.
Downcasting floats.


In [55]:
#test2
colts_2024 = PbP("IND", 2024, "Personal project/nfl_pbp_column_categories_v3 (1).csv", "playbyplay2")

In [56]:
colts_2024.evaluate()

2024 done.
Downcasting floats.
2015 done.
Downcasting floats.
